# BRCA2021 Marker-Gene Audit — Training Set

Evaluates whether the VAE preserves cancer-subtype identity and marker biology
in generated single-cell profiles, relative to matched training-set real cells
(BreastCancer SunnyWu 2021).

**Marker panels:** 9 cancer-subtype markers (ER+ / HER2+ / TNBC) · 9 immune-cell
markers (T cell / B cell / Macrophage) · 2 housekeeping negative controls.

**Analysis structure**

| # | Section | Visulisation |
|---|---------|-----------------|
| 1 | Joint Seurat HVG UMAP | Overal UMAP: real vs generated umap |
| 2 | Cell-type & sample UMAP | UMAPs of real cells colored by two celltype label |
| 3 | Per-gene expression overlays | UMAPs of marker gene expression: real vs generated |
| 4 | Expression distributions | Boxplots of marker gene expression in different cancer subtypes |
| 5 | Marker-positive cell fraction | Barplots of cell fraction of marker gene significantly expressed: real vs generated |
| 6 | Global gene mean correlation | Scatter plot of per-gene mean log1p expression across all cells: real and generated |


In [ ]:
import glob
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
from pathlib import Path
from scipy import sparse

PROJECT_ROOT = Path('/share/home/xiaojied/bulk2scDiff')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from guided_diffusion.cell_datasets_loader import load_VAE, read_preprocessed_adata, read_sample_ids_file

DATA_DIR       = '/share/data/transcriptomics/single cell/curated_mini_umi/BreastCancer_SunnyWu_2021_AnnData.h5ad'
SAMPLE_IDS_PATH = PROJECT_ROOT / 'output/sample_splits/brca2021_manual/train_samples.txt'
GENERATED_GLOB  = str(PROJECT_ROOT / 'output/simulated_samples/brca2021_pseudobulk_1M_alltrain_train_*.npz')
VAE_PATH        = PROJECT_ROOT / 'output/checkpoint/AE/brca2021_VAE/model_seed=1234_step=199999.pt'

SAMPLE_KEY     = 'SampleID'
GROUP_KEY      = 'subtype'
HIDDEN_DIM     = 128
DECODE_BATCH   = 1024
SEED           = 1234

# UMAP uses Seurat dispersion-based HVG on the full transcriptome
UMAP_PER_SAMPLE = 1000  # cells kept per sample for the UMAP matrices (real and generated separately).
UMAP_N_PCS      = 20    # PCs used in sc.pp.neighbors after HVG -> scale -> PCA
UMAP_N_NBRS     = 10    # k-NN neighbours (matches the per-sample UMAP notebook)


GROUP_ORDER = ["ER+", "HER2+", "TNBC"]

MARKER_PANELS = {
    "ER+":   ["ESR1", "PGR", "GATA3"],
    "HER2+": ["ERBB2", "GRB7", "PGAP3"],
    "TNBC":  ["KRT5", "KRT14", "EGFR"],
}
IMMUNE_MARKER_PANELS = {
    "T cell":     ["CD3D", "CD8A", "CD4"],
    "B cell":     ["CD19", "MS4A1", "CD79A"],
    "Macrophage": ["CD68", "CSF1R", "LYZ"],
}
NEGATIVE_CONTROL_GENES = ["ACTB", "HPRT1"]

ALL_IMMUNE_GENES  = [g for panel in IMMUNE_MARKER_PANELS.values() for g in panel]
ALL_PANEL_GENES   = [g for panel in MARKER_PANELS.values() for g in panel] + NEGATIVE_CONTROL_GENES + ALL_IMMUNE_GENES
GENE_TO_PANEL     = {g: grp for grp, panel in MARKER_PANELS.items() for g in panel}
for _g in NEGATIVE_CONTROL_GENES:
    GENE_TO_PANEL[_g] = "neg_control"
for _immune_type, _immune_genes in IMMUNE_MARKER_PANELS.items():
    for _g in _immune_genes:
        GENE_TO_PANEL[_g] = _immune_type
GENE_DISPLAY_ORDER = (
    MARKER_PANELS["ER+"] + MARKER_PANELS["HER2+"] + MARKER_PANELS["TNBC"] + NEGATIVE_CONTROL_GENES
    + IMMUNE_MARKER_PANELS["T cell"] + IMMUNE_MARKER_PANELS["B cell"] + IMMUNE_MARKER_PANELS["Macrophage"]
)

SUBTYPE_COLORS = {"ER+": "#E91E63", "HER2+": "#FF9800", "TNBC": "#2196F3"}
SOURCE_COLORS  = {"real": "#1976D2", "generated": "#F4511E"}
ROW_COLORS     = {
    "ER+ markers":        "#E91E63",
    "HER2+ markers":      "#FF9800",
    "TNBC markers":       "#2196F3",
    "Neg controls":       "#9E9E9E",
    "T cell markers":     "#4CAF50",
    "B cell markers":     "#9C27B0",
    "Macrophage markers": "#FF5722",
}

sc.settings.verbosity = 1
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
print("Configuration loaded.")

In [ ]:
def scalar_from_npz(value):
    array = np.asarray(value)
    return array.item() if array.shape == () else array.reshape(-1)[0]


def gather_sparse_rows_and_cols(matrix, row_mask, col_indices):
    view = matrix[row_mask][:, col_indices]
    if sparse.issparse(view):
        view = view.toarray()
    return np.asarray(view, dtype=np.float32)


def decode_marker_columns_with_loaded_vae(latent_data, vae, marker_indices, batch_size, device):
    latent_data = np.asarray(latent_data, dtype=np.float32)
    batches = []
    with torch.no_grad():
        for start in range(0, latent_data.shape[0], batch_size):
            batch = latent_data[start:start + batch_size]
            decoded = vae(torch.from_numpy(batch).to(device), return_decoded=True)
            batches.append(decoded[:, marker_indices].cpu().numpy().astype(np.float32))
    return np.concatenate(batches, axis=0)


def decode_all_genes(latent_data, vae, batch_size, device):
    latent_data = np.asarray(latent_data, dtype=np.float32)
    batches = []
    with torch.no_grad():
        for start in range(0, latent_data.shape[0], batch_size):
            batch = latent_data[start:start + batch_size]
            decoded = vae(torch.from_numpy(batch).to(device), return_decoded=True)
            batches.append(decoded.cpu().numpy().astype(np.float32))
    return np.concatenate(batches, axis=0)


def build_umap_seurat_hvg(matrix, obs, gene_names, n_neighbors=10, n_pcs=20):
    """Build UMAP using Seurat dispersion-based HVG on the joint real+generated pool.

    Dispersion is normalised within mean bins, so genes that vary only due to a
    systematic real-vs-generated offset do not pass min_disp=0.5 and are excluded
    from PCA. adata.raw stores full-transcriptome log1p values pre-HVG-filter,
    used by the per-gene UMAP overlay cell.
    """
    from sklearn.decomposition import PCA as _PCA  # local import; only needed here
    mat = np.nan_to_num(np.clip(np.asarray(matrix, dtype=np.float32), -100, 100))
    adata = ad.AnnData(mat)
    adata.var_names = [str(g) for g in gene_names]
    adata.obs = obs.reset_index(drop=True).copy()
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
    n_hvg = int(adata.var["highly_variable"].sum())
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable].copy()
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=n_neighbors,
                    n_pcs=min(n_pcs, adata.obsm["X_pca"].shape[1]))
    sc.tl.umap(adata)
    print(f"  Seurat HVG: {n_hvg} genes -> PCA -> UMAP")
    return adata


print("Helper functions defined.")

In [ ]:
# 1. Load real AnnData, filter to test samples
adata = read_preprocessed_adata(DATA_DIR)
sample_values = adata.obs[SAMPLE_KEY].astype(str).to_numpy()
group_values  = adata.obs[GROUP_KEY].astype(str).to_numpy()

allowed_ids = read_sample_ids_file(str(SAMPLE_IDS_PATH))
if allowed_ids is not None:
    keep_mask     = np.isin(sample_values, list(allowed_ids))
    adata         = adata[keep_mask].copy()
    sample_values = adata.obs[SAMPLE_KEY].astype(str).to_numpy()
    group_values  = adata.obs[GROUP_KEY].astype(str).to_numpy()

sample_to_group = {
    s_id: str(np.unique(group_values[sample_values == s_id])[0])
    for s_id in np.unique(sample_values)
}

# 2. Map 11 panel genes to AnnData column indices
available_genes    = np.asarray(adata.var_names.astype(str))
gene_to_index      = {gene: idx for idx, gene in enumerate(available_genes)}
all_genes_in_adata = [g for g in ALL_PANEL_GENES if g in gene_to_index]
missing_genes      = [g for g in ALL_PANEL_GENES if g not in gene_to_index]
if missing_genes:
    print(f"WARNING: genes not found in dataset: {missing_genes}")
all_gene_indices = [gene_to_index[g] for g in all_genes_in_adata]

# 3. Locate generated .npz files
generated_paths = sorted(glob.glob(GENERATED_GLOB))
path_by_sample  = {}
for path_str in generated_paths:
    with np.load(path_str, allow_pickle=False) as f:
        sid = str(scalar_from_npz(f["source_sample_id"]))
    if allowed_ids is None or sid in allowed_ids:
        path_by_sample[sid] = path_str

# 4. Load VAE
device = "cuda" if torch.cuda.is_available() else "cpu"
vae = load_VAE(vae_path=str(VAE_PATH), num_gene=adata.n_vars, hidden_dim=HIDDEN_DIM, device=device)
vae.eval()

# 5. Extract 11 marker genes per sample → full per-cell arrays + summary store
#    (used for bar charts, boxplots, sparsity — NOT for UMAP)
subtype_gene_store = {
    st: {gene: {"real": [], "generated": []} for gene in all_genes_in_adata}
    for st in GROUP_ORDER
}
real_cells_list, gen_cells_list = [], []
real_obs_rows,   gen_obs_rows   = [], []

print(f"Decoding {len(all_genes_in_adata)} marker genes for {len(path_by_sample)} samples...")
for sample_id, path in sorted(path_by_sample.items()):
    if sample_id not in sample_to_group:
        continue
    group_name  = sample_to_group[sample_id]
    sample_mask = sample_values == sample_id

    with np.load(path, allow_pickle=False) as f:
        generated_latent = np.asarray(f["cell_gen"], dtype=np.float32)

    real_expr = gather_sparse_rows_and_cols(adata.X, sample_mask, all_gene_indices)
    gen_expr  = decode_marker_columns_with_loaded_vae(
        generated_latent, vae, all_gene_indices, DECODE_BATCH, device
    )

    real_cells_list.append(real_expr)
    gen_cells_list.append(gen_expr)
    real_obs_rows.extend([{"sample_id": sample_id, "subtype": group_name, "source": "real"}]      * len(real_expr))
    gen_obs_rows.extend( [{"sample_id": sample_id, "subtype": group_name, "source": "generated"}] * len(gen_expr))

    for col_idx, gene in enumerate(all_genes_in_adata):
        subtype_gene_store[group_name][gene]["real"].append(real_expr[:, col_idx])
        subtype_gene_store[group_name][gene]["generated"].append(gen_expr[:, col_idx])

    print(f"  {sample_id} ({group_name}): {int(sample_mask.sum())} real, {generated_latent.shape[0]} generated")

# 6. Concatenate full marker-gene arrays
real_matrix_full = np.concatenate(real_cells_list, axis=0)   # (N_real, 11)
gen_matrix_full  = np.concatenate(gen_cells_list,  axis=0)   # (N_gen,  11)
real_obs_full    = pd.DataFrame(real_obs_rows)
gen_obs_full     = pd.DataFrame(gen_obs_rows)

# 7. Build summary DataFrame (pct_expressing + mean for bar chart / sparsity)
marker_summary_rows = []
for cell_subtype in GROUP_ORDER:
    for gene in all_genes_in_adata:
        for source in ("real", "generated"):
            arrays = subtype_gene_store[cell_subtype][gene][source]
            if not arrays:
                continue
            values = np.concatenate(arrays).astype(np.float32)
            marker_summary_rows.append({
                "cell_subtype":    cell_subtype,
                "gene":            gene,
                "marker_group":    GENE_TO_PANEL[gene],
                "source":          source,
                "n_cells":         int(values.size),
                "mean_expression": float(values.mean()),
                "pct_expressing":  float((values > 0).mean()),
            })
marker_summary_df = pd.DataFrame(marker_summary_rows)

print(f"\nMarker-gene analysis data ready.")
print(f"  real: {real_matrix_full.shape},  generated: {gen_matrix_full.shape}")
print(f"  summary rows: {len(marker_summary_df)}")

In [ ]:
# Full-transcriptome matrices for Seurat HVG UMAP
# Each test sample contributes at most UMAP_PER_SAMPLE real and UMAP_PER_SAMPLE generated cells.
# These are separate from real_matrix_full / gen_matrix_full (marker genes) which are
# used for bar charts, boxplots, and sparsity — those cells are unchanged.

rng_umap_fg   = np.random.default_rng(SEED + 10)
full_gene_names = list(adata.var_names.astype(str))
_has_celltype_major = 'celltype_major' in adata.obs.columns
_has_celltype_minor = 'celltype_minor' in adata.obs.columns

real_umap_fg_blocks, gen_umap_fg_blocks = [], []
real_umap_fg_obs,    gen_umap_fg_obs    = [], []

print(f"Loading full-gene expression for Seurat HVG UMAP ({len(path_by_sample)} samples, "
      f"\u2264{UMAP_PER_SAMPLE} cells/sample)...")

for sample_id, path in sorted(path_by_sample.items()):
    if sample_id not in sample_to_group:
        continue
    group_name = sample_to_group[sample_id]
    smask      = sample_values == sample_id

    # Real cells: full transcriptome (log1p-normalized, same as adata.X)
    real_s = adata.X[smask]
    if sparse.issparse(real_s):
        real_s = real_s.toarray()
    real_s = np.asarray(real_s, dtype=np.float32)

    # Generated cells: decode all genes from stored latents
    with np.load(path, allow_pickle=False) as f:
        lat_s = np.asarray(f["cell_gen"], dtype=np.float32)
    gen_s = decode_all_genes(lat_s, vae, DECODE_BATCH, device)

    # Per-sample subsample
    nr = min(real_s.shape[0], UMAP_PER_SAMPLE)
    ng = min(gen_s.shape[0],  UMAP_PER_SAMPLE)
    real_idx = rng_umap_fg.choice(real_s.shape[0], nr, replace=False)
    gen_idx  = rng_umap_fg.choice(gen_s.shape[0],  ng, replace=False)
    real_s   = real_s[real_idx]
    gen_s    = gen_s[gen_idx]

    if _has_celltype_major or _has_celltype_minor:
        smask_pos   = np.where(smask)[0]
        global_idx  = smask_pos[real_idx]
        row_base    = {"sample_id": sample_id, "subtype": group_name, "source": "real"}
        major_vals  = adata.obs["celltype_major"].values[global_idx] if _has_celltype_major else [None] * nr
        minor_vals  = adata.obs["celltype_minor"].values[global_idx] if _has_celltype_minor else [None] * nr
        real_umap_fg_obs.extend([
            {**row_base, "celltype_major": maj, "celltype_minor": mn}
            for maj, mn in zip(major_vals, minor_vals)
        ])
    else:
        real_umap_fg_obs.extend(
            [{"sample_id": sample_id, "subtype": group_name, "source": "real"}] * nr
        )
    gen_umap_fg_obs.extend(
        [{"sample_id": sample_id, "subtype": group_name, "source": "generated"}] * ng
    )

    real_umap_fg_blocks.append(real_s)
    gen_umap_fg_blocks.append(gen_s)
    print(f"  {sample_id} ({group_name}): {nr} real, {ng} gen  [{adata.n_vars} genes]")

real_matrix_umap_fg = np.concatenate(real_umap_fg_blocks, axis=0)
gen_matrix_umap_fg  = np.concatenate(gen_umap_fg_blocks,  axis=0)
real_obs_umap_fg    = pd.DataFrame(real_umap_fg_obs)
gen_obs_umap_fg     = pd.DataFrame(gen_umap_fg_obs)

print(f"\nFull-gene UMAP matrices ready.")
print(f"  real: {real_matrix_umap_fg.shape},  gen: {gen_matrix_umap_fg.shape}")

## 1. Joint Seurat HVG UMAP — Source and Subtype Overview

Embeds real and generated cells jointly in a space constructed from Seurat
dispersion-selected HVGs across the full transcriptome. Three panels: mixed
(real + generated coloured by source), real-only, and generated-only, all coloured
by cancer subtype.

**Pass criterion:** generated cells intermix with real cells within each subtype
cluster; subtype boundaries are preserved in the generated-only panel.

**Known limitation:** a joint UMAP can absorb systematic real–generated shifts
into apparent overlap, masking calibration failures — cross-validate with
Sections 5 and 6.

In [ ]:
# Joint Seurat HVG UMAP
# Built from the full transcriptome using Seurat dispersion-based HVG selection.
# Three panels: real + generated together | real only | generated only.
# All panels coloured by subtype so cluster geometry is directly comparable.

combined_matrix_fg = np.vstack([real_matrix_umap_fg, gen_matrix_umap_fg])
combined_obs_fg = pd.concat(
    [real_obs_umap_fg.assign(source="real"), gen_obs_umap_fg.assign(source="generated")],
    ignore_index=True,
)
combined_obs_fg["subtype"] = pd.Categorical(combined_obs_fg["subtype"], categories=GROUP_ORDER)
combined_obs_fg["source"]  = pd.Categorical(combined_obs_fg["source"])

adata_joint = build_umap_seurat_hvg(
    combined_matrix_fg, combined_obs_fg, full_gene_names,
    n_neighbors=UMAP_N_NBRS, n_pcs=UMAP_N_PCS,
)
print(f"  {adata_joint.n_obs} cells embedded.")

coords          = adata_joint.obsm["X_umap"]
source_vals     = adata_joint.obs["source"].values
subtype_vals    = adata_joint.obs["subtype"].values
real_mask_joint = source_vals == "real"
gen_mask_joint  = ~real_mask_joint

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1 — real + generated together, coloured by source
for label, color in SOURCE_COLORS.items():
    mask = source_vals == label
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=color, s=4,
                    alpha=0.4, label=label, rasterized=True)
axes[0].set_title("Real + Generated (together)", fontsize=12, fontweight="bold")
axes[0].legend(markerscale=3, fontsize=9, frameon=False)

# Panel 2 — real cells only, coloured by subtype
for label, color in SUBTYPE_COLORS.items():
    mask = real_mask_joint & (subtype_vals == label)
    axes[1].scatter(coords[mask, 0], coords[mask, 1], c=color, s=4,
                    alpha=0.5, label=label, rasterized=True)
axes[1].set_title("Real cells only", fontsize=12, fontweight="bold")
axes[1].legend(markerscale=3, fontsize=9, frameon=False)

# Panel 3 — generated cells only, coloured by subtype
for label, color in SUBTYPE_COLORS.items():
    mask = gen_mask_joint & (subtype_vals == label)
    axes[2].scatter(coords[mask, 0], coords[mask, 1], c=color, s=4,
                    alpha=0.5, label=label, rasterized=True)
axes[2].set_title("Generated cells only", fontsize=12, fontweight="bold")
axes[2].legend(markerscale=3, fontsize=9, frameon=False)

for ax in axes:
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.set_aspect("equal", "datalim")

fig.suptitle(
    "Breast Cancer UMAPs Marker Genes Audit(Seurat HVG) ",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.show()

## 2. Joint UMAP — Author Cell Type and Sample Identity

Four panels using the same joint Seurat HVG embedding:

- **celltype_major** (real) — broad lineage labels; checks whether major cell
  compartments (epithelial, immune, stromal) occupy distinct UMAP regions.
- **celltype_minor** (real) — fine-grained subtypes within each major lineage;
  reveals internal structure that major labels collapse.
- **sample_id** (real vs generated) — assesses whether generation is uniform
  across donors or collapses to a subset of the latent neighbourhood.

Both cell-type panels are N/A-safe: if a label column is absent from the
dataset, the panel renders a placeholder message.

In [ ]:
# Joint Seurat HVG UMAP — author cell type and sample identity
# 2x2 layout:
#   [0,0] Real cells — celltype_major
#   [0,1] Real cells — celltype_minor
#   [1,0] Real cells — sample_id
#   [1,1] Generated cells — sample_id

coords          = adata_joint.obsm["X_umap"]
source_vals     = adata_joint.obs["source"].values
real_mask_joint = source_vals == "real"
gen_mask_joint  = ~real_mask_joint
coords_real     = coords[real_mask_joint]
coords_gen      = coords[gen_mask_joint]

all_sample_ids = sorted(set(adata_joint.obs["sample_id"].values))
cmap_sid       = plt.cm.get_cmap("tab20", max(len(all_sample_ids), 1))
sid_color_map  = {sid: cmap_sid(i) for i, sid in enumerate(all_sample_ids)}

# 60-color palette: tab20 + tab20b + tab20c — sufficient for most minor cell-type panels
_PALETTE = list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors) + list(plt.cm.tab20c.colors)
_NULL_STRS = {'nan', 'None', 'NaN', 'none', 'NA', 'na', ''}

def _celltype_panel(ax, ct_col, title):
    if ct_col not in adata_joint.obs.columns:
        ax.text(0.5, 0.5, f"'{ct_col}' not in obs",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_title(title + " (N/A)", fontsize=11, fontweight="bold")
        return
    cts    = adata_joint.obs[ct_col].values[real_mask_joint]
    unique = sorted(set(str(c) for c in cts if str(c) not in _NULL_STRS))
    ct_cmap = {ct: _PALETTE[i % len(_PALETTE)] for i, ct in enumerate(unique)}
    for ct in unique:
        mask = np.array([str(c) == ct for c in cts])
        ax.scatter(coords_real[mask, 0], coords_real[mask, 1],
                   c=[ct_cmap[ct]], s=4, alpha=0.5, label=ct, rasterized=True)
    ncol = 2 if len(unique) > 20 else 1
    ax.legend(markerscale=3, fontsize=6, frameon=False,
              loc="upper left", bbox_to_anchor=(1.01, 1), ncol=ncol)
    ax.set_title(title, fontsize=11, fontweight="bold")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

_celltype_panel(axes[0, 0], "celltype_major", "Real — celltype_major")
_celltype_panel(axes[0, 1], "celltype_minor", "Real — celltype_minor")

# [1,0] real cells by sample_id
sample_vals_real = adata_joint.obs["sample_id"].values[real_mask_joint]
for sid in all_sample_ids:
    mask = sample_vals_real == sid
    if mask.any():
        axes[1, 0].scatter(coords_real[mask, 0], coords_real[mask, 1],
                           c=[sid_color_map[sid]], s=4, alpha=0.5, label=sid, rasterized=True)
axes[1, 0].set_title("Real — sample ID", fontsize=11, fontweight="bold")
axes[1, 0].legend(markerscale=3, fontsize=6, frameon=False,
                  loc="upper left", bbox_to_anchor=(1.01, 1), ncol=1)

# [1,1] generated cells by sample_id
sample_vals_gen = adata_joint.obs["sample_id"].values[gen_mask_joint]
for sid in all_sample_ids:
    mask = sample_vals_gen == sid
    if mask.any():
        axes[1, 1].scatter(coords_gen[mask, 0], coords_gen[mask, 1],
                           c=[sid_color_map[sid]], s=4, alpha=0.5, label=sid, rasterized=True)
axes[1, 1].set_title("Generated — sample ID", fontsize=11, fontweight="bold")
axes[1, 1].legend(markerscale=3, fontsize=6, frameon=False,
                  loc="upper left", bbox_to_anchor=(1.01, 1), ncol=1)

for ax in axes.flat:
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.set_aspect("equal", "datalim")

fig.suptitle(
    "Joint UMAP (Seurat HVG)  —  Author Cell Type & Sample Identity",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

## 3. Per-Gene Expression Overlays on Joint UMAP

Renders each marker gene as a log1p colour overlay (real left, generated right).
Colour scale is shared per gene and capped at the real 99th percentile to prevent
rare generated outliers from compressing the real range.

Rows 1–3: cancer-subtype markers. Rows 4–6: immune-cell markers. Row 7: negative
controls. Genes absent from the dataset are silently hidden.

**Pass criterion:** high-expression zones in generated panels co-localise with
the same UMAP regions as in real panels. Diffuse or shifted gradients indicate
expression infidelity.

In [ ]:
# Per-gene expression on the joint Seurat HVG UMAP — real (left) vs generated (right)
# Layout: 7 rows x 6 cols (3 genes per group x 2 panels each: real | generated)
# Rows: ER+, HER2+, TNBC, T cell, B cell, Macrophage, Neg controls
# Colour = raw log1p expression (from adata_joint.raw, pre-HVG-filter, pre-scale).
# Shared vmin/vmax per gene (computed from real cells' 99th-percentile).

gene_rows = [
    ("ER+ markers",        MARKER_PANELS["ER+"]),
    ("HER2+ markers",      MARKER_PANELS["HER2+"]),
    ("TNBC markers",       MARKER_PANELS["TNBC"]),
    ("T cell markers",     IMMUNE_MARKER_PANELS["T cell"]),
    ("B cell markers",     IMMUNE_MARKER_PANELS["B cell"]),
    ("Macrophage markers", IMMUNE_MARKER_PANELS["Macrophage"]),
    ("Neg controls",       NEGATIVE_CONTROL_GENES),
]

raw_var_names   = list(adata_joint.raw.var_names)
raw_gene_idx    = {g: i for i, g in enumerate(raw_var_names)}
raw_X_src       = adata_joint.raw.X
real_mask_joint = adata_joint.obs["source"].values == "real"
gen_mask_joint  = ~real_mask_joint

coords_real = coords[real_mask_joint]
coords_gen  = coords[gen_mask_joint]

# Pre-extract only the marker gene columns we will actually plot
marker_expr_cache = {}
for _, genes in gene_rows:
    for gene in genes:
        if gene in raw_gene_idx:
            col = raw_X_src[:, raw_gene_idx[gene]]
            if hasattr(col, "toarray"):
                col = col.toarray().ravel()
            marker_expr_cache[gene] = np.asarray(col, dtype=np.float32)

n_genes_per_row = max(len(genes) for _, genes in gene_rows)
n_cols = n_genes_per_row * 2  # real + generated per gene
n_rows = len(gene_rows)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows), constrained_layout=True)
fig.suptitle(
    "Marker Gene Expression on Seurat HVG UMAP  |  Real (left) vs Generated (right)\n"
    "colour = raw log1p  |  rows 1-3: cancer subtypes  |  rows 4-6: immune cell types  |  row 7: neg controls",
    fontsize=13, fontweight="bold",
)

for row_idx, (row_label, genes) in enumerate(gene_rows):
    row_color = ROW_COLORS[row_label]
    for gene_col_idx in range(n_genes_per_row):
        ax_real = axes[row_idx, gene_col_idx * 2]
        ax_gen  = axes[row_idx, gene_col_idx * 2 + 1]

        if gene_col_idx >= len(genes):
            ax_real.set_visible(False)
            ax_gen.set_visible(False)
            continue

        gene = genes[gene_col_idx]
        if gene not in marker_expr_cache:
            ax_real.set_visible(False)
            ax_gen.set_visible(False)
            continue

        expr      = marker_expr_cache[gene]
        expr_real = expr[real_mask_joint]
        expr_gen  = expr[gen_mask_joint]
        vmax = float(np.nanpercentile(expr_real, 99)) or 1.0

        # Real panel (left)
        sc_real = ax_real.scatter(
            coords_real[:, 0], coords_real[:, 1], c=expr_real, cmap="magma",
            vmin=0, vmax=vmax, s=2, alpha=0.5, rasterized=True,
        )
        ax_real.set_title(f"{gene}\n(real)", fontsize=10, fontweight="bold", color=row_color)
        ax_real.set_xticks([]); ax_real.set_yticks([])
        ax_real.set_aspect("equal", "datalim")
        plt.colorbar(sc_real, ax=ax_real, fraction=0.046, pad=0.02)
        for spine in ax_real.spines.values():
            spine.set_edgecolor(row_color); spine.set_linewidth(2.0); spine.set_visible(True)

        # Generated panel (right)
        sc_gen = ax_gen.scatter(
            coords_gen[:, 0], coords_gen[:, 1], c=expr_gen, cmap="magma",
            vmin=0, vmax=vmax, s=2, alpha=0.5, rasterized=True,
        )
        ax_gen.set_title(f"{gene}\n(generated)", fontsize=10, fontweight="bold", color=row_color)
        ax_gen.set_xticks([]); ax_gen.set_yticks([])
        ax_gen.set_aspect("equal", "datalim")
        plt.colorbar(sc_gen, ax=ax_gen, fraction=0.046, pad=0.02)
        for spine in ax_gen.spines.values():
            spine.set_edgecolor(row_color); spine.set_linewidth(2.0); spine.set_visible(True)

plt.show()

## 4. Per-Gene Expression Distributions

**Cancer markers (9 genes):** four groups per gene — Real/Generated × Same/Other
cancer subtype.

**Immune markers (9 genes):** real vs generated across all cells. Non-trivial
expression is expected only if the training cohort contains a meaningful immune
fraction; near-zero generated expression when real expression is detectable
indicates immune-marker dropout.

**Negative controls (ACTB, HPRT1):** should be uniformly expressed at comparable
levels in real and generated — large discrepancies suggest a global calibration
error rather than marker-specific failure.

In [ ]:
# Per-marker-gene boxplots
# Cancer markers (9 genes): 4 boxes — Real/Gen x Same/Other cancer subtype
#   same-subtype box should sit markedly higher than other-subtype box
# Immune cell markers (9 genes) + Neg controls (2 genes): 2 boxes — Real vs Generated (all cells)

MAX_CELLS_PER_BOX = 2000
rng_box = np.random.default_rng(SEED + 99)

cancer_gene_list = [g for st in GROUP_ORDER for g in MARKER_PANELS[st]]
immune_gene_list = ALL_IMMUNE_GENES
neg_ctrl_list    = NEGATIVE_CONTROL_GENES
all_box_genes    = cancer_gene_list + immune_gene_list + neg_ctrl_list
cancer_gene_set  = set(cancer_gene_list)
neg_ctrl_set     = set(neg_ctrl_list)

# Filter to genes present in the loaded marker matrix
all_box_genes_present = [g for g in all_box_genes if g in all_genes_in_adata]
n_genes    = len(all_box_genes_present)
n_cols_bp  = 4
n_rows_bp  = (n_genes + n_cols_bp - 1) // n_cols_bp

fig, axes = plt.subplots(n_rows_bp, n_cols_bp,
                          figsize=(5 * n_cols_bp, 4.5 * n_rows_bp),
                          constrained_layout=True)
axes_flat = axes.flatten()

real_subtypes_arr = real_obs_full["subtype"].values
gen_subtypes_arr  = gen_obs_full["subtype"].values

def _subsample(matrix, mask, col):
    idx = np.where(mask)[0]
    if len(idx) > MAX_CELLS_PER_BOX:
        idx = rng_box.choice(idx, MAX_CELLS_PER_BOX, replace=False)
    return matrix[idx, col]

for ax_idx, gene in enumerate(all_box_genes_present):
    ax    = axes_flat[ax_idx]
    g_col = all_genes_in_adata.index(gene)

    if gene in cancer_gene_set:
        # 4-box: real/gen x same/other cancer subtype
        gene_subtype = GENE_TO_PANEL[gene]
        data = [
            _subsample(real_matrix_full, real_subtypes_arr == gene_subtype, g_col),
            _subsample(real_matrix_full, real_subtypes_arr != gene_subtype, g_col),
            _subsample(gen_matrix_full,  gen_subtypes_arr  == gene_subtype, g_col),
            _subsample(gen_matrix_full,  gen_subtypes_arr  != gene_subtype, g_col),
        ]
        lbs  = ["Real\n(same)", "Real\n(other)", "Gen\n(same)", "Gen\n(other)"]
        clrs = ["#1976D2", "#90CAF9", "#F4511E", "#FFAB91"]
        title = f"{gene}  ({gene_subtype} marker)"
        tcol  = SUBTYPE_COLORS[gene_subtype]
    else:
        # 2-box: real vs generated, all cells
        all_real = np.ones(real_matrix_full.shape[0], dtype=bool)
        all_gen  = np.ones(gen_matrix_full.shape[0],  dtype=bool)
        data = [
            _subsample(real_matrix_full, all_real, g_col),
            _subsample(gen_matrix_full,  all_gen,  g_col),
        ]
        lbs  = ["Real\n(all)", "Generated\n(all)"]
        clrs = [SOURCE_COLORS["real"], SOURCE_COLORS["generated"]]
        if gene in neg_ctrl_set:
            title = f"{gene}  (neg control)"
            tcol  = ROW_COLORS["Neg controls"]
        else:
            immune_type = GENE_TO_PANEL[gene]
            title = f"{gene}  ({immune_type} marker)"
            tcol  = ROW_COLORS.get(f"{immune_type} markers", "#333333")

    bp = ax.boxplot(
        data, patch_artist=True, notch=False,
        medianprops={"color": "black", "linewidth": 1.5},
        whiskerprops={"linewidth": 1.0},
        flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    )
    for patch, color in zip(bp["boxes"], clrs):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)

    ax.set_xticks(list(range(1, len(lbs) + 1)))
    ax.set_xticklabels(lbs, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold", color=tcol)
    ax.set_ylabel("log1p expression", fontsize=8)
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")

for ax_idx in range(n_genes, len(axes_flat)):
    axes_flat[ax_idx].set_visible(False)

fig.suptitle(
    "Per-Marker-Gene Expression Distributions\n"
    "Cancer markers: Real/Gen x Same/Other subtype  "
    "| Immune markers & neg controls: Real vs Generated (all cells)",
    fontsize=13, fontweight="bold",
)
plt.show()

## 5. Marker-Positive Cell Fraction

For each marker gene, computes the fraction of cells with log1p expression >
`EXPR_THRESHOLD` (default 0.5) in real and generated separately. One bar-chart
panel per cell-identity category (cancer subtypes then immune cell types).
Genes absent from the dataset are skipped with a warning.

**Diagnostic use:** quantifies whether the model preserves marker detection
rates. A large real–generated gap signals expression collapse or dropout
inflation in the generated cells. A near-zero real fraction for immune markers
indicates the training cohort lacks that cell type — generated near-zero is
then expected, not a failure.

In [ ]:
# Marker-positive cell fraction comparison
# Threshold for 'marker-positive' (log1p space) — adjust as needed
EXPR_THRESHOLD = 0.5

all_marker_categories = {**MARKER_PANELS, **IMMUNE_MARKER_PANELS}

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True, sharey=True)
axes_list = axes.flatten().tolist()

for ax_idx, (ax, (category, genes)) in enumerate(zip(axes_list, all_marker_categories.items())):
    genes_present = [g for g in genes if g in all_genes_in_adata]
    missing = [g for g in genes if g not in all_genes_in_adata]
    if missing:
        print(f"WARNING [{category}]: genes not found in dataset, skipping: {missing}")
    if not genes_present:
        ax.text(0.5, 0.5, f"No genes available\nfor {category}",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_title(category, fontsize=11, fontweight="bold")
        continue

    g_indices  = [all_genes_in_adata.index(g) for g in genes_present]
    real_fracs = [float((real_matrix_full[:, i] > EXPR_THRESHOLD).mean()) for i in g_indices]
    gen_fracs  = [float((gen_matrix_full[:,  i] > EXPR_THRESHOLD).mean()) for i in g_indices]

    x     = np.arange(len(genes_present))
    width = 0.38
    ax.bar(x - width / 2, real_fracs, width,
           color=SOURCE_COLORS["real"], alpha=0.85, label="real")
    ax.bar(x + width / 2, gen_fracs,  width,
           color=SOURCE_COLORS["generated"], alpha=0.85, label="generated", hatch="//")

    ax.set_xticks(x)
    ax.set_xticklabels(genes_present, rotation=45, ha="right", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_title(category, fontsize=11, fontweight="bold")
    if ax_idx % 3 == 0:
        ax.set_ylabel(f"Fraction expressing > {EXPR_THRESHOLD} (log1p)")
    ax.legend(fontsize=9, frameon=False)

fig.suptitle(
    f"Marker-Positive Cell Fraction  (threshold = {EXPR_THRESHOLD} log1p)\n"
    "Real vs Generated across all cells  "
    "| row 1: cancer subtypes  |  row 2: immune cell types",
    fontsize=13, fontweight="bold",
)
plt.show()

## 6. Global Gene Mean Correlation

Computes per-gene mean log1p expression across all cells (real and generated,
drawn from the full-transcriptome UMAP pool). Reports Pearson r and Spearman ρ;
scatter plot includes a y = x reference line and highlights all marker genes.

**Diagnostic use:** high r/ρ is necessary but not sufficient — a model that
scales all gene means by a constant factor can achieve high correlation while
distorting relative expression. Inspect the scatter for systematic bias above
or below y = x, and check whether marker genes deviate from the bulk trend.

In [ ]:
# Global gene mean correlation
from scipy.stats import pearsonr, spearmanr

def _sparse_safe_mean(mat):
    if sparse.issparse(mat):
        return np.asarray(mat.mean(axis=0)).ravel()
    return np.asarray(mat, dtype=np.float32).mean(axis=0)

real_gene_means = _sparse_safe_mean(real_matrix_umap_fg)
gen_gene_means  = _sparse_safe_mean(gen_matrix_umap_fg)

valid = np.isfinite(real_gene_means) & np.isfinite(gen_gene_means)
r,   _ = pearsonr( real_gene_means[valid], gen_gene_means[valid])
rho, _ = spearmanr(real_gene_means[valid], gen_gene_means[valid])
print(f"Pearson r  = {r:.4f}")
print(f"Spearman \u03c1 = {rho:.4f}")

# One representative label per group to avoid clutter
label_genes = set()
for panel in MARKER_PANELS.values():
    label_genes.add(panel[0])
for panel in IMMUNE_MARKER_PANELS.values():
    label_genes.add(panel[0])
label_genes.update(NEGATIVE_CONTROL_GENES)

gene_name_to_umap_idx = {g: i for i, g in enumerate(full_gene_names)}
highlight_genes = [g for g in ALL_PANEL_GENES if g in gene_name_to_umap_idx]
highlight_set   = set(highlight_genes)
highlight_mask  = np.array([g in highlight_set for g in full_gene_names], dtype=bool)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

# Background genes
bg = ~highlight_mask & valid
ax.scatter(real_gene_means[bg], gen_gene_means[bg],
           c="#CCCCCC", s=5, alpha=0.35, rasterized=True,
           label=f"other genes (n={bg.sum():,})")

# Highlighted marker genes
hi = highlight_mask & valid
ax.scatter(real_gene_means[hi], gen_gene_means[hi],
           c="#E91E63", s=40, alpha=0.9, zorder=5,
           label=f"marker genes (n={hi.sum()})")

# Label only lead gene per group
for i in np.where(hi)[0]:
    gene = full_gene_names[i]
    if gene in label_genes:
        ax.annotate(gene, (real_gene_means[i], gen_gene_means[i]),
                    fontsize=7, xytext=(4, 3), textcoords="offset points",
                    zorder=6)

# y = x reference line
lim_min = min(real_gene_means[valid].min(), gen_gene_means[valid].min())
lim_max = max(real_gene_means[valid].max(), gen_gene_means[valid].max())
ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", linewidth=1.0, label="y = x")

ax.set_xlabel("Real — mean log1p expression per gene", fontsize=10)
ax.set_ylabel("Generated — mean log1p expression per gene", fontsize=10)
ax.set_title(
    f"Global Gene Mean Correlation\n"
    f"Pearson r = {r:.4f}  |  Spearman \u03c1 = {rho:.4f}",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()